# Reproduction: Amortized Variational Inference for Partial-Label Learning

**Paper:** Fuchs & Klein, ICML 2026 — https://arxiv.org/abs/2510.21300

**Code:** https://github.com/mathefuchs/vi-pll

This notebook reproduces the ViPll method from scratch.

In [ ]:
# Install dependencies
!pip install torch torchvision numpy scipy scikit-learn matplotlib

In [ ]:
import math, warnings, os, json, time, numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from scipy.optimize import minimize
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold
from copy import deepcopy

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 1. MLP Base Classifier

In [ ]:
class MLP(nn.Module):
    def __init__(self, m_features, l_classes):
        super().__init__()
        self.m_features = m_features
        self.mlp = nn.Sequential(
            nn.Linear(m_features, 300), nn.ReLU(),
            nn.BatchNorm1d(300), nn.Linear(300, l_classes),
        )
    def forward(self, inputs):
        inputs = inputs.view(-1, self.m_features)
        logits = self.mlp(inputs)
        return F.softmax(logits, dim=1), logits, torch.tensor(0.0)

## 2. CVAE (Conditional Variational Auto-Encoder)

In [ ]:
class ConditionalEncoder(nn.Module):
    def __init__(self, input_dim, y_dim, latent_dim, hidden_dim=128):
        super().__init__()
        self.fc1 = nn.Sequential(nn.Linear(input_dim + y_dim, hidden_dim), nn.ReLU())
        self.fc2_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc2_logvar = nn.Linear(hidden_dim, latent_dim)
    def forward(self, x, y):
        h = self.fc1(torch.cat((x, y), dim=1))
        return self.fc2_mu(h), self.fc2_logvar(h)

class ConditionalDecoder(nn.Module):
    def __init__(self, latent_dim, y_dim, output_dim, hidden_dim=128):
        super().__init__()
        self.recon = nn.Sequential(
            nn.Linear(latent_dim + y_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, output_dim), nn.Hardsigmoid(),
        )
    def forward(self, z, y):
        return self.recon(torch.cat((z, y), dim=-1))

class CVAE(nn.Module):
    def __init__(self, input_dim, y_dim, latent_dim, hidden_dim):
        super().__init__()
        self.encoder = ConditionalEncoder(input_dim, y_dim, latent_dim, hidden_dim)
        self.decoder = ConditionalDecoder(latent_dim, y_dim, input_dim, hidden_dim)
    def forward(self, x, y):
        mu, logvar = self.encoder(x, y)
        std = torch.exp(0.5 * logvar)
        z = mu + torch.randn_like(std) * std
        return self.decoder(z, y), mu, logvar

## 3. ViPll — Full Implementation

In [ ]:
class ViPll:
    """ViPll: Variational Inference for Partial-Label Learning (ICML 2026)"""
    
    def __init__(self, rng, num_classes, input_dim, device, lmbd=10.0, prior_coef=0.0, num_epoch=1001, batch_size=64, debug=True):
        self.rng, self.num_classes, self.input_dim = rng, num_classes, input_dim
        self.device, self.lmbd, self.prior_coef = device, lmbd, prior_coef
        self.num_epoch, self.batch_size, self.debug = num_epoch, batch_size, debug
        
        # Two MLPs in ensemble (paper uses model averaging)
        self.model = MLP(input_dim, num_classes).to(device)
        self.model2 = deepcopy(self.model).to(device)
        
        # CVAE: larger latent/hidden for 784-dim inputs (MNIST), smaller otherwise
        large = (input_dim == 784)
        self.p_x_y = CVAE(input_dim, num_classes, 
                         latent_dim=128 if large else 32,
                         hidden_dim=512 if large else 128).to(device)
    
    def _max_entropy_prior(self, partial_targets):
        """Section 4.3: Maximum-entropy prior satisfying candidate constraints."""
        def entropy(x):
            x = np.clip(x, 1e-10, 1)
            return -np.sum(x * np.log(x))
        n = partial_targets.shape[0]
        min_f = partial_targets[partial_targets.sum(1) == 1].sum(0) / n
        max_f = partial_targets.sum(0) / n
        x0 = 0.5 * (max_f + min_f)
        x0 /= x0.sum()
        res = minimize(lambda x: -entropy(x), x0,
                      bounds=list(zip(min_f, max_f)),
                      constraints=[{'type': 'eq', 'fun': lambda x: x.sum() - 1}])
        return res.x
    
    def _warmup_cvae(self, inputs, targets):
        """Phase 2: 500-epoch CVAE warm-up (Algorithm 1 Lines 4-10)."""
        opt = torch.optim.AdamW(self.p_x_y.parameters(), lr=1e-4)
        loader = DataLoader(TensorDataset(torch.tensor(inputs, dtype=torch.float32),
                                          torch.tensor(targets, dtype=torch.float32)),
                          batch_size=self.batch_size, shuffle=True, drop_last=True)
        losses = []
        for ep in range(501):
            for xb, sb in loader:
                xb, sb = xb.to(self.device), sb.to(self.device)
                xr, mu, lv = self.p_x_y(xb, sb)
                rl = F.mse_loss(xr, xb)
                kl = -0.5 * torch.mean(1 + lv - mu**2 - lv.exp())
                losses.append(math.sqrt(rl.item()))
                opt.zero_grad(); (rl + kl).backward(); opt.step()
            if self.debug and ep % 100 == 0:
                print(f'  CVAE [{ep}/500] RMSE: {np.mean(losses[-10:]):.4f}')
        ema = losses[0]
        for l in losses[1:]: ema = 0.95 * ema + 0.05 * l
        return float(ema)
    
    def _log_p_x_given_y(self, x, y, ns, sigma):
        """Importance-weighted estimate of log P(X|Y) (Eq. 21)."""
        self.p_x_y.eval()
        with torch.no_grad():
            mu, lv = self.p_x_y.encoder(x, y)
            z = mu + torch.randn(ns, *mu.shape, device=x.device) * torch.exp(0.5 * lv)
            xr = self.p_x_y.decoder(z, y.view(1, *y.shape).repeat(ns, 1, 1))
            mse = ((x.view(1, *x.shape).repeat(ns, 1, 1) - xr)**2).mean(-1)
            l1 = -0.5 * (mse / sigma**2 + math.log(2 * math.pi * sigma**2))
            l2 = -0.5 * (z**2 + math.log(2 * math.pi)).mean(-1)
            l3 = -0.5 * ((z - mu)**2 / torch.exp(lv) + lv + math.log(2 * math.pi)).mean(-1)
            return torch.logsumexp(l1 + 0.01 * (l2 - l3), 0) - math.log(ns)
    
    def fit(self, inputs, partial_targets):
        """Full ViPll training (Algorithm 1)."""
        # Phase 1: Initialize
        prior = torch.tensor(self._max_entropy_prior(partial_targets), dtype=torch.float32, device=self.device)
        sigma = self._warmup_cvae(inputs, partial_targets)
        self.p_x_y.eval()
        
        # Data
        x_t = torch.tensor(inputs, dtype=torch.float32)
        s_t = torch.tensor(partial_targets, dtype=torch.float32)
        idx = torch.arange(len(inputs), dtype=torch.long)
        cy = torch.tensor(partial_targets, dtype=torch.float32)
        loader = DataLoader(TensorDataset(idx, x_t, s_t, cy), batch_size=self.batch_size, shuffle=True, drop_last=True)
        
        opt = torch.optim.AdamW(list(self.model.parameters()) + list(self.model2.parameters()))
        opt_pxy = torch.optim.AdamW(self.p_x_y.parameters(), lr=1e-4)
        
        # Phase 3: Main training (T=1000 epochs, Lines 11-22)
        self.model.train(); self.model2.train()
        for ep in range(self.num_epoch):
            losses = []
            for ib, xb, sb, cyb in loader:
                xb, sb, cyb = xb.to(self.device), sb.to(self.device), cyb.to(self.device)
                
                # Q(Y|X) = Dirichlet(alpha), alpha = 1 + softplus(logits) (Eq. 1)
                logits = 0.5 * (self.model(xb + 0.01*torch.randn_like(xb))[1] +
                               self.model2(xb + 0.01*torch.randn_like(xb))[1])
                alpha = 1 + F.softplus(logits)
                q_samples = torch.distributions.Dirichlet(alpha).rsample((10,))  # b=10
                
                # ELBO terms (Eq. 10)
                lq = torch.mean(torch.sum(q_samples * torch.log(q_samples + 1e-10), -1), 0)
                lp_y = torch.mean(torch.sum(q_samples * torch.log(prior + 1e-10), -1), 0)
                
                with torch.no_grad():
                    xr = torch.repeat_interleave(xb, self.num_classes, 0)
                    yr = torch.eye(self.num_classes, device=xb.device).repeat(xb.shape[0], 1)
                    lp_xy = self._log_p_x_given_y(xr, yr, 10, sigma).view(xb.shape[0], self.num_classes)
                lp_xy = torch.mean(torch.sum(q_samples * lp_xy, -1), 0)
                lp_s = torch.mean(torch.sum(q_samples * torch.log(sb + 1e-10), -1), 0)
                
                loss = torch.mean(lq - self.prior_coef * lp_y - self.lmbd * lp_xy - lp_s)
                
                # Update CVAE
                self.p_x_y.train()
                xr2, mu, lv = self.p_x_y(xb, cyb)
                loss += F.mse_loss(xr2, xb) - 0.5 * torch.mean(1 + lv - mu**2 - lv.exp())
                
                losses.append(loss.item())
                opt.zero_grad(); opt_pxy.zero_grad(); loss.backward(); opt.step(); opt_pxy.step()
                
                # Update labeling (Line 22)
                with torch.no_grad():
                    qb = alpha / alpha.sum(1, keepdim=True)
                    np_ = (qb + 1e-10) * sb; np_ /= np_.sum(1, keepdim=True)
                    cy[ib] = (np_ / np_.max(1, keepdim=True).values).cpu()
            
            if self.debug and ep % 200 == 0:
                print(f'  [{ep}/{self.num_epoch}] loss: {np.mean(losses[-10:]):.4f}')
        return self.predict(inputs)
    
    def predict(self, inputs):
        self.model.eval(); self.model2.eval()
        loader = DataLoader(TensorDataset(torch.tensor(inputs, dtype=torch.float32)),
                          batch_size=self.batch_size, shuffle=False)
        probs = []
        with torch.no_grad():
            for xb in loader:
                xb = xb[0].to(self.device)
                p = 1 + F.softplus(0.5 * (self.model(xb)[1] + self.model2(xb)[1]))
                probs.append((p / p.sum(1, keepdim=True)).cpu().numpy())
        return np.vstack(probs)

## 4. ViAblation (Discriminative Baseline)

In [ ]:
class ViAblation:
    """ViPll ablation: removes generative P(X|Y) and P(Y) modeling."""
    def __init__(self, rng, num_classes, input_dim, device, num_epoch=1001, batch_size=64, debug=True):
        self.model = MLP(input_dim, num_classes).to(device)
        self.device, self.num_epoch, self.batch_size, self.debug = device, num_epoch, batch_size, debug
    
    def fit(self, inputs, partial_targets):
        x_t = torch.tensor(inputs, dtype=torch.float32)
        idx = torch.arange(len(inputs), dtype=torch.int32)
        pyx = torch.tensor(partial_targets, dtype=torch.float32)
        pyx /= pyx.sum(1, keepdim=True)
        loader = DataLoader(TensorDataset(idx, x_t, torch.tensor(partial_targets, dtype=torch.float32), pyx),
                          batch_size=self.batch_size, shuffle=True, drop_last=True)
        opt = torch.optim.AdamW(self.model.parameters())
        self.model.train()
        for ep in range(self.num_epoch):
            losses = []
            for i, xb, sb, pb in loader:
                xb, sb, pb = xb.to(self.device), sb.to(self.device), pb.to(self.device)
                alpha = 1 + F.softplus(self.model(xb)[1])
                qs = torch.distributions.Dirichlet(alpha).rsample((10,))
                lq = torch.mean(torch.sum(qs * torch.log(qs + 1e-10), -1), 0)
                lp = torch.mean(torch.sum(qs * torch.log(pb + 1e-10), -1), 0)
                ls = torch.mean(torch.sum(qs * torch.log(sb + 1e-10), -1), 0)
                loss = torch.mean(lq - lp - ls)
                losses.append(loss.item())
                opt.zero_grad(); loss.backward(); opt.step()
                with torch.no_grad():
                    qb = alpha / alpha.sum(1, keepdim=True)
                    np_ = (qb + 1e-10) * sb; np_ /= np_.sum(1, keepdim=True)
                    pyx[i] = np_.cpu()
            if self.debug and ep % 200 == 0:
                print(f'  Ablation [{ep}/{self.num_epoch}] loss: {np.mean(losses):.4f}')
        return self.predict(inputs)
    
    def predict(self, inputs):
        self.model.eval()
        loader = DataLoader(TensorDataset(torch.tensor(inputs, dtype=torch.float32)), batch_size=self.batch_size, shuffle=False)
        probs = []
        with torch.no_grad():
            for xb in loader:
                xb = xb[0].to(self.device)
                p = 1 + F.softplus(self.model(xb)[1])
                probs.append((p / p.sum(1, keepdim=True)).cpu().numpy())
        return np.vstack(probs)

## 5. Experimental Setup & Data Loading

In [ ]:
def add_instance_dependent_noise(x_data, y_data, num_classes):
    """Add partial labels using instance-dependent noise strategy from Section 5.2."""
    # Train a noise model
    xf = x_data.reshape(x_data.shape[0], -1).copy()
    mn, mx = xf.min(0), xf.max(0)
    xf = (xf - mn) / np.clip(mx - mn, 1e-10, None)
    
    model = nn.Sequential(nn.Linear(xf.shape[1], 300), nn.ReLU(),
                          nn.Linear(300, y_data.shape[1]), nn.Softmax(dim=1))
    opt = torch.optim.SGD(model.parameters())
    loader = DataLoader(TensorDataset(torch.tensor(xf), torch.tensor(y_data)), batch_size=64, shuffle=True, drop_last=True)
    model.train()
    for _ in range(20):
        for xb, yb in loader:
            opt.zero_grad(); torch.mean(torch.sum((yb - model(xb))**2, 1)).backward(); opt.step()
    
    model.eval()
    with torch.no_grad():
        probs = np.vstack([model(xb[0]).numpy() for xb in DataLoader(TensorDataset(torch.tensor(xf)), batch_size=64, shuffle=False)])
    
    # Class imbalance
    imb = np.array([0.025**((i+1)/num_classes) for i in range(num_classes)])
    np.random.shuffle(imb); imb /= imb.max()
    
    fp = (1 - y_data) * probs
    fp = 0.3 * fp / np.clip(fp.max(1, keepdims=True), 1e-10, None) + 0.7 * imb
    fp = fp / np.clip(fp.mean(1, keepdims=True), 1e-10, None)
    fp = np.clip(fp, 0.001, 0.999)
    
    mask = torch.distributions.Binomial(1, torch.tensor(fp)).sample().numpy().astype(bool)
    y_noisy = y_data.copy()
    y_noisy[mask] = 1
    return y_noisy

def load_real_world(name):
    """Load a real-world PLL dataset from .mat file."""
    from scipy.io import loadmat
    import urllib.request
    path = f'{name}.mat'
    if not os.path.exists(path):
        urllib.request.urlretrieve(
            f'https://raw.githubusercontent.com/mathefuchs/vi-pll/main/data/realworld-datasets/{name}.mat', path)
    raw = loadmat(path)
    x = np.array(raw['data']).squeeze()
    yp = np.array(raw['partial_target'].T).squeeze()
    yt = np.argmax(np.array(raw['target'].T).squeeze(), 1)
    return x, yp, yt

def load_supervised(name):
    """Load supervised dataset and add partial labels."""
    import torchvision
    ds = {'mnist': torchvision.datasets.MNIST, 'fmnist': torchvision.datasets.FashionMNIST,
          'kmnist': torchvision.datasets.KMNIST}
    train = ds[name]('./data', train=True, transform=torchvision.transforms.ToTensor(), download=True)
    test = ds[name]('./data', train=False, transform=torchvision.transforms.ToTensor(), download=True)
    xtr = (train.data.view(-1, 784).numpy() / 255).copy()
    xte = (test.data.view(-1, 784).numpy() / 255).copy()
    ytr, yte = train.targets.numpy(), test.targets.numpy()
    k = len(np.unique(ytr))
    Y = np.zeros((xtr.shape[0], k), dtype=int)
    for i, y in enumerate(ytr): Y[i, y] = 1
    Yn = add_instance_dependent_noise(xtr, Y, k)
    return xtr, xte, Yn, ytr, yte

def run_experiment(dataset, method_class, method_name, n_folds=5, **kwargs):
    """Run experiment with cross-validation."""
    print(f'\n=== {dataset} - {method_name} ===')
    
    if dataset in ['bird-song', 'lost', 'mir-flickr', 'msrc-v2', 'yahoo-news']:
        x, y, yt = load_real_world(dataset)
        params = {'bird-song': (2.5, 5.0), 'lost': (50.0, 0.0)}
        lmbd, prior = params.get(dataset, (10.0, 0.0))
        skf = StratifiedKFold(n_folds, shuffle=True, random_state=42)
        splits = list(enumerate(skf.split(x, yt)))
    else:
        xtr, xte, ytr, ytrt, ytet = load_supervised(dataset)
        lmbd, prior = 200.0, 0.0
        # Use train/test split
        x, yt = xtr, ytrt
        splits = [(np.arange(len(xtr)), np.arange(len(xte)))]
        
    accs = []
    for fi, (tr_idx, te_idx) in splits:
        np.random.seed(42 + fi); torch.manual_seed(42 + fi)
        rng = np.random.Generator(np.random.PCG64(42 + fi))
        
        if dataset in ['bird-song', 'lost', 'mir-flickr', 'msrc-v2', 'yahoo-news']:
            xt, yt_tr = x[tr_idx], y[tr_idx]
            xv = x[te_idx]
            yv = yt[te_idx]
        else:
            xt, yt_tr = x[tr_idx], ytr[tr_idx]
            xv = xte
            yv = ytet
        
        # Normalize
        mn, mx = xt.min(0), xt.max(0)
        xt = (xt - mn) / np.clip(mx - mn, 1e-10, None)
        xv = (xv - mn) / np.clip(mx - mn, 1e-10, None)
        
        clf = method_class(rng, yt_tr.shape[1], xt.shape[1], device, lmbd=lmbd, prior_coef=prior, debug=True)
        t0 = time.time()
        probs = clf.fit(xt, yt_tr)
        pred = clf.predict(xv)
        acc = accuracy_score(yv, np.argmax(pred, 1))
        accs.append(acc)
        print(f'  Fold {fi}: {acc:.4f} ({time.time()-t0:.1f}s)')
    
    print(f'  Result: {np.mean(accs):.4f} +/- {np.std(accs):.4f}')
    return {'accs': accs, 'mean': float(np.mean(accs)), 'std': float(np.std(accs))}

## 6. Run Experiments

In [ ]:
# Run ViPll on bird-song (real-world)
result = run_experiment('bird-song', ViPll, 'ViPll', n_folds=1)  # Set to 5 for full reproduction

In [ ]:
# Run ViPll on lost (real-world)
result = run_experiment('lost', ViPll, 'ViPll', n_folds=1)

In [ ]:
# Run ViPll on MNIST (synthetic)
result = run_experiment('mnist', ViPll, 'ViPll', n_folds=1)

In [ ]:
# Run ablation on bird-song
result = run_experiment('bird-song', ViAblation, 'ViAblation', n_folds=1)

## 7. Summary & Verification against paper claims

| Claim | Expected | Observed |
|---|---|---|
| C1: Dirichlet posterior | q(y\|x,s) = Dir(·; α_φ) | Implemented via softplus + Dirichlet |
| C2: CVAE + max-entropy prior | P(X\|Y) via CVAE, P(Y) via Eq. 22 | Both implemented |
| C3: Three-phase training | Init → 500 ep CVAE → 1000 ep main | Implemented |
| C4: bird-song = 76.15% | 76.15% ±1.56 | TBD (run full 5-fold) |
| C5: CIFAR-10 = 96.64% | 96.64% ±3.92 | TBD (run full 5-fold) |
| C6: Ablation degrades | ViPll > ablation | TBD |

**Note:** The paper's full results are pre-computed and available at https://github.com/mathefuchs/vi-pll